In [ ]:
# Solutions live in solutions/. Run from the repository root so imports and paths work.
import os, pathlib, sys
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "workshop").is_dir())
os.chdir(_root)
sys.path.insert(0, str(_root))

# Module 5: Package and publish your widget

**Time:** 20 minutes

A widget in a notebook folder helps only you. A **package** lets anyone run
`pip install your-widget`. With anywidget, the JavaScript ships **inside** the
Python package: one package on PyPI, one version number, no npm.

1. Look at the package template
2. Make a package from a widget folder
3. Build a wheel and check what is inside
4. Test the **installed** package, and break it on purpose
5. Publish with GitHub Actions and PyPI trusted publishing

## 1. The template

In [ ]:
import pathlib

template = pathlib.Path("template/widget-package")
for path in sorted(template.rglob("*")):
    if path.is_file():
        print(path.relative_to(template))

- `pyproject.toml`: name, version, dependencies (`anywidget>=0.11`)
- `src/<module>/`: your widget folder goes here, with its `static/` files
- `tests/test_package.py`: checks that the JavaScript and CSS ship in the package
- `.github/workflows/publish.yml`: test, build, and publish on every version tag

## 2. Make a package

Use the reviewed `RangeLabeler`, or your own widget from Module 4
(`my_widgets/<name>`). Choose a `dist_name` that is free on PyPI
(check `https://pypi.org/project/<name>/`).

In [ ]:
from workshop.packaging import build, make_package, test_installed, wheel_files

project = make_package(
    widget_dir="workshop/range_labeler",
    dist_name="range-labeler-demo",
    description="Mark labelled time ranges on a signal in Jupyter.",
    author="Workshop Participant",
    dependencies=["numpy", "pandas"],
    overwrite=True,
)
for path in sorted(project.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(path.relative_to(project))

In [ ]:
print((project / "pyproject.toml").read_text())

## 3. Build

`build` creates two files in `dist/`: a **wheel** (`.whl`, what `pip install`
uses) and a source archive (`.tar.gz`). This runs offline.

In [ ]:
artifacts = build(project)
for path in artifacts:
    print(f"{path.name:55s} {path.stat().st_size / 1024:6.1f} KB")

A wheel is a zip file. The JavaScript and CSS must be inside it:

In [ ]:
wheel = next(p for p in artifacts if p.suffix == ".whl")
for name in wheel_files(wheel):
    print(name)

## 4. Test the installed package

`test_installed` installs the wheel into a temporary folder and runs the tests
**against the installed copy**, not your source folder. A missing JavaScript
file fails here, not on a colleague's computer.

In [ ]:
assert test_installed(project)

### Break it on purpose

The most common packaging bug: `_esm` as a relative string path (checklist E1).
It works in your folder and fails everywhere else. Watch the test catch it.

In [ ]:
init = next((project / "src").glob("*/__init__.py"))
original = init.read_text()

init.write_text(original.replace('_STATIC / "widget.js"', '"static/widget.js"'))
build(project)
print("With the bug, tests pass:", test_installed(project))

init.write_text(original)
build(project)
print("After the fix, tests pass:", test_installed(project))

## 5. Publish

Publishing is a demonstration today (it needs accounts and internet). The
template already contains everything:

In [ ]:
print((project / ".github" / "workflows" / "publish.yml").read_text())

**One-time setup**

1. Push the project folder to a new GitHub repository.
2. On [pypi.org](https://pypi.org), go to *Your projects → Publishing* and add
   a **pending publisher**: your repository, workflow `publish.yml`,
   environment `pypi`. This is **trusted publishing**: no API token is stored.
3. In the GitHub repository settings, create an environment called `pypi`.
4. Practise first on [test.pypi.org](https://test.pypi.org) with the same steps.

**Every release**

1. Change `version` in `pyproject.toml`.
2. `git tag v0.1.0 && git push --tags`
3. GitHub Actions runs the tests, builds, checks that `static/` is in the
   wheel, and publishes.

**Also useful**

- **conda-forge** (popular on clusters): after the first PyPI release, run
  `grayskull pypi <name>` and open a pull request to `conda-forge/staged-recipes`.
- **Users need `anywidget` in the Jupyter server environment** (see
  `resources/troubleshooting.md`). On JupyterHub, an admin installs it once.
- **Get it seen:** add your widget to the anywidget community gallery
  (a pull request to `manzt/built-with-anywidget`).

## Wrap-up

- **Use first:** look in [`resources/widget_catalog.md`](resources/widget_catalog.md).
- **Build second:** anywidget + the prompt template + the review checklist.
- **Ship it:** the package template, tested against the installed wheel.

Sharing without a kernel, apps, and further reading:
[`resources/next_steps.md`](resources/next_steps.md).

**Please fill in the feedback form.** Thank you!